In [ ]:
# Importe de librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, OrdinalEncoder

# Carga de Datos

In [ ]:
!wget https://raw.githubusercontent.com/Octaviochavez/analitica_clientes/main/data/dataset_clientes.csv

In [ ]:
# Lectura de datos sucios del dataset desde directorio local data
data = pd.read_csv('dataset_clientes.csv')
data.head()

# 1. Análisis exploratorio inicial

## Análisis de tipos de datos de cada categoría (columna)

In [ ]:
# Muestra características del dataset
data.info()

##  Análisis de valores en columnas cuantitativas
* Identificamos que las columnas "tiene_tarjeta_credito" y "abandono" debe ser booleano y no entero, ya que sus valores o son 1 o son 0.

In [ ]:
# Muestra estadísticas descriptivas iniciales de variables numéricas
data.drop(columns=["id_cliente","codigo_postal"]).describe()

## Análisis de valores en columnas cualitativas
* columnas ordinales: "uso_app" y "tipo_plan"
* columnas nominales: "genero", "region", "estado_civil", "canal_registro" y "dia_semana_registro"

In [ ]:
# Muestra estadísticas descriptivas iniciales de variables categóricas
data.describe(include="object")

## Análisis de valores únicos en variables cualitativas

##### Variable Region

In [ ]:
data.region.unique()

##### Variable Estado Civil

In [ ]:
data.estado_civil.unique()

##### Variable Tipo Plan

In [ ]:
data.tipo_plan.unique()

##### Variable Uso App

In [ ]:
data.uso_app.unique()

##### Variable Canal Registro

In [ ]:
data.canal_registro.unique()

##### Variable Género

In [ ]:
data.genero.unique()

##### Variable Día Semana Registro

In [ ]:
data.dia_semana_registro.unique()

## Análisis de valores nulos
* Determinamos que "ingreso_mensual" y "score_crediticio" contienen la misma cantidad de nulos, y "gasto_mensual" tiene una cantidad apenas superior.

In [ ]:
# Obtención de datos nulos
data.isnull().sum()

## Análisis de valores duplicados


In [ ]:
# Obtención de duplicados
data.duplicated().sum()
data[data.duplicated(keep=False)].sort_values(by='id_cliente')

## Análisis de valores atípicos
* Se observa que "tiene_tarjeta_credito" y "abandono" no son candidatos para la búsqueda de atípicos, ya que cuentan como categóricos (booleanos).
Además se puede identificar a las columnas "ingreso_mensual", "gasto_mensual", "deuda_total" y "score_crediticio" con alta probabilidad de tener datos atípicos.

In [ ]:
# Arrays de variables numéricas para análisis de atípicos
registro_atipicos=["edad",	"ingreso_mensual",	"gasto_mensual",	"deuda_total",	"score_crediticio",	"antiguedad_meses",	"frecuencia_compra",	"ultima_compra_dias",	"num_productos",	"hora_registro"]

# Gráfico BoxPlot
fig, axes = plt.subplots(1, 10, figsize=(20,20))
axes = axes.flatten()
for i, col in enumerate(registro_atipicos):
  data[col].plot(kind='box',figsize=(12,12), ax=axes[i])
  axes[i].tick_params(axis="x", labelrotation=45)

plt.suptitle("Visualización de Outliers", fontsize=16, fontweight="bold")
plt.tight_layout()

plt.show()

## Análisis de distribución inicial de columnas numéricas
* Se generan visualizaciones de cómo es la distribución inicial de principales variables numéricas que contienen datos sucios y serán sometidas a limpieza.

In [ ]:
data['ingreso_mensual'].hist()
plt.title('Distribución inicial de Ingreso Mensual antes de limpieza', fontsize=14, fontweight="bold")
plt.show()

In [ ]:
data['gasto_mensual'].hist()
plt.title('Distribución inicial de Gasto Mensual antes de limpieza', fontsize=14, fontweight="bold")
plt.show()

In [ ]:
data['deuda_total'].hist()
plt.title('Distribución inicial de Deuda Total antes de limpieza', fontsize=14, fontweight="bold")
plt.show()

In [ ]:
data['score_crediticio'].hist()
plt.title('Distribución inicial de Score Crediticio antes de limpieza', fontsize=14, fontweight="bold")
plt.show()

# 2. Limpieza y Transformación de Datos

## Clase para winsorizar los datos
* Esto con el fin de limpiar los outliers.

In [ ]:
class Winsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, limits=(0.05, 0.05)):
        self.limits = limits

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        # Calculamos y guardamos los límites matemáticos AQUÍ (Fase de aprendizaje)
        self.lower_bounds_ = X_df.quantile(self.limits[0])
        self.upper_bounds_ = X_df.quantile(1 - self.limits[1])
        self.columns_ = X_df.columns if isinstance(X, pd.DataFrame) else np.arange(X.shape[1])
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X, columns=self.columns_).copy()
        X_df = X_df.astype("float64")
        # Aplicamos los límites guardados
        for col in self.columns_:
            X_df[col] = np.clip(X_df[col], self.lower_bounds_[col], self.upper_bounds_[col])
        return X_df

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return np.array(self.columns_)
        else:
            return np.array(input_features)

## Tratamiento de duplicados

In [ ]:
# Define función para eliminar duplicados
def eliminar_duplicados(X):
  return X.drop_duplicates()

# Ingeniería de Características (Feature Engineering)

In [ ]:
def agregar_variables(data : pd.DataFrame):

  data = data.copy()

  # razón de endeudamiento
  data["ratio_endeudamiento"] = data["deuda_total"] / data["ingreso_mensual"]

  # porcentaje de gasto respecto al ingreso
  data['porcentaje_gasto'] = data['gasto_mensual'] / data['ingreso_mensual']

  return data

In [ ]:
data = agregar_variables(data) # Agrega nuevas variables al dataset original
data.head()

In [ ]:
data['tiene_tarjeta_credito'] = data['tiene_tarjeta_credito'].astype('category') # Convierte Variable 'Tiene Tarjeta de Crédito' de numérica a categórica
data['abandono'] = data['abandono'].astype('category') # Convierte Variable Abandono de numérica a categórica

## Construcción de arrays para procesamiento

In [ ]:
numerical_features = ["gasto_mensual", "score_crediticio", "ingreso_mensual", "deuda_total", 'porcentaje_gasto', 'ratio_endeudamiento', "edad", "antiguedad_meses", "frecuencia_compra", "ultima_compra_dias", "num_productos", "hora_registro"] # Define listado de variables numéricas
categorical_nominales = ['abandono', "tiene_tarjeta_credito", "genero", "region", "estado_civil", "canal_registro","dia_semana_registro"] # Define listado de variables categóricas nominales
categorical_ordinales = ["tipo_plan", "uso_app"] # Define listado de variables categóricas ordinales
date_time_features = ["fecha_registro"] # Define listado de variables de fecha y hora
orden_tipo_plan = ['Basico', 'Estandar', 'Premium'] # Define orden para variable ordinal tipo_plan
orden_uso_app = ['Bajo', 'Medio', 'Alto'] # Define orden para variable ordinal uso_app

## Pipelines para cada tipo de conjunto


In [ ]:
# Define pipeline para variables numéricas
pipeline_numerical_features = Pipeline(steps=[
    ('winsorizer', Winsorizer(limits=(0.05, 0.05))), # Aplica Winsorización para limitar outliers al 5%
    ('imputer', SimpleImputer(strategy='mean')), # Imputa valores faltantes con el promedio
    ('scaler', StandardScaler()) # Escala características numéricas
])

In [ ]:
# Define pipeline para variables categóricas nominales
pipeline_nominales = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Imputa valores faltantes con la moda
    ('onehot', OneHotEncoder(handle_unknown='ignore')) # Aplica codificación OneHotEncoder para variables nominales
])

In [ ]:
# Define pipeline para variables categóricas ordinales
pipeline_ordinales = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Imputa valores faltantes con la moda
    ('ordinal', OrdinalEncoder(categories=[orden_tipo_plan, orden_uso_app])) # Aplica codificación OrdinalEncoder para variables ordinales con orden definido
])

## Integración de pipelines de transformación


In [ ]:
# Combina pipelines para aplicar transformaciones específicas a cada tipo de variable
preprocesador = ColumnTransformer(
    transformers=[
        ('num_limpios', pipeline_numerical_features, numerical_features),
        ('cat_nom', pipeline_nominales, categorical_nominales),
        ('cat_ord', pipeline_ordinales, categorical_ordinales),
    ],
    remainder='drop' # Elimina columnas no especificadas
)

## Pipeline para limpieza


In [ ]:
# Define pipeline de limpieza que incluye eliminación de duplicados y preprocesamiento específico para cada tipo de variable
pipeline_limpieza = Pipeline(
    steps=[
        ("duplicados", FunctionTransformer(eliminar_duplicados)),
        ("preprocesamiento", preprocesador)
    ]
)

In [ ]:
# Aplica pipeline de limpieza al dataset y obtiene el resultado transformado
data_transformada = pd.DataFrame(
    pipeline_limpieza.fit_transform(data),
    columns=pipeline_limpieza.named_steps["preprocesamiento"].get_feature_names_out()
)
data_transformada.columns = data_transformada.columns.str.replace("num_limpios__", "")
data_transformada.columns = data_transformada.columns.str.replace("cat_nom__", "")
data_transformada.columns = data_transformada.columns.str.replace("cat_ord__", "")
data_transformada.head()

# 3. Resultados

In [ ]:
# Muestra características del dataset después de la limpieza y transformación
data_transformada.info()

In [ ]:
# Obtención de datos nulos después de la limpieza y transformación
data_transformada.isnull().sum()

In [ ]:
# Obtención de duplicados después de la limpieza y transformación
data_transformada.duplicated().sum()

In [ ]:
# Medidas estadísticas descriptivas después de la limpieza y transformación
data_transformada.describe()

## Resultados de codificación ordinal

In [ ]:
# Conteo de valores de variable Tipo Plan después de codificación ordinal y limpieza
data_transformada.tipo_plan.value_counts()

In [ ]:
# Conteo de valores de variable Uso App después de codificación ordinal y limpieza
data_transformada.uso_app.value_counts()

## Análisis de distribución final

In [ ]:
data_transformada['ingreso_mensual'].hist()
plt.title('Distribución de Ingreso Mensual después de limpieza y transformaicón', fontsize=14, fontweight="bold")
plt.show()

In [ ]:
data_transformada['deuda_total'].hist()
plt.title('Distribución de Deuda Total después de limpieza y transformaicón', fontsize=14, fontweight="bold")
plt.show()

In [ ]:
data_transformada['gasto_mensual'].hist()
plt.title('Distribución de Gasto Mensual después de limpieza y transformaicón', fontsize=14, fontweight="bold")
plt.show()

In [ ]:
data_transformada['score_crediticio'].hist()
plt.title('Distribución de Score Crediticio después de limpieza y transformaicón', fontsize=14, fontweight="bold")
plt.show()

In [ ]:
# Exporta dataset limpio a nuevo archivo CSV en el directorio actual
data_transformada.to_csv('dataset_clientes_limpio.csv', index=False, encoding='utf-8')